In [ ]:
import os # used to get env variables

import json # used for changing json formatting (see last cell)

import requests # used to get api response data

import pandas as pd # used for data transformation

import psycopg as pg # used for db connection and querying

from dotenv import load_dotenv # used to load variables from the .env file

In [2]:
load_dotenv() # loads env variables in .env file


True

In [3]:
# .env variables and other parameters
API_KEY = os.getenv("API_KEY")
API_HOST = os.getenv("API_HOST")
API_URL = os.getenv("API_URL")
LEAGUE_ID = 47


In [ ]:
# API params
querystring = {"leagueid":LEAGUE_ID}

headers = {
    "x-rapidapi-key": API_KEY,
    "x-rapidapi-host": API_HOST,
    "Content-Type": "application/json"
}

# EXTRACT

In [6]:
# Sends request to API
response = requests.get(
    # Needs URL
    API_URL,
    # header variable from above to find API
    headers=headers, 
    # parameters 
    params=querystring)

In [7]:
# Sends response to payload in json format
payload = response.json()

In [8]:
payload

{'status': 'success',
 'response': {'standing': [{'name': 'Manchester City',
    'shortName': 'Man City',
    'id': 8456,
    'pageUrl': '/teams/8456/overview/manchester-city',
    'deduction': None,
    'ongoing': None,
    'played': 5,
    'wins': 5,
    'draws': 0,
    'losses': 0,
    'scoresStr': '13-5',
    'goalConDiff': 8,
    'pts': 15,
    'idx': 1,
    'qualColor': '#2AD572'},
   {'name': 'Arsenal',
    'shortName': 'Arsenal',
    'id': 9825,
    'pageUrl': '/teams/9825/overview/arsenal',
    'deduction': None,
    'ongoing': None,
    'played': 5,
    'wins': 4,
    'draws': 0,
    'losses': 1,
    'scoresStr': '8-4',
    'goalConDiff': 4,
    'pts': 12,
    'idx': 2,
    'qualColor': '#2AD572'},
   {'name': 'Brighton & Hove Albion',
    'shortName': 'Brighton',
    'id': 10204,
    'pageUrl': '/teams/10204/overview/brighton-hove-albion',
    'deduction': None,
    'ongoing': None,
    'played': 5,
    'wins': 3,
    'draws': 1,
    'losses': 1,
    'scoresStr': '16-5',
   

In [9]:
standings = payload["response"]['standing']
standings

[{'name': 'Manchester City',
  'shortName': 'Man City',
  'id': 8456,
  'pageUrl': '/teams/8456/overview/manchester-city',
  'deduction': None,
  'ongoing': None,
  'played': 5,
  'wins': 5,
  'draws': 0,
  'losses': 0,
  'scoresStr': '13-5',
  'goalConDiff': 8,
  'pts': 15,
  'idx': 1,
  'qualColor': '#2AD572'},
 {'name': 'Arsenal',
  'shortName': 'Arsenal',
  'id': 9825,
  'pageUrl': '/teams/9825/overview/arsenal',
  'deduction': None,
  'ongoing': None,
  'played': 5,
  'wins': 4,
  'draws': 0,
  'losses': 1,
  'scoresStr': '8-4',
  'goalConDiff': 4,
  'pts': 12,
  'idx': 2,
  'qualColor': '#2AD572'},
 {'name': 'Brighton & Hove Albion',
  'shortName': 'Brighton',
  'id': 10204,
  'pageUrl': '/teams/10204/overview/brighton-hove-albion',
  'deduction': None,
  'ongoing': None,
  'played': 5,
  'wins': 3,
  'draws': 1,
  'losses': 1,
  'scoresStr': '16-5',
  'goalConDiff': 11,
  'pts': 10,
  'idx': 3,
  'qualColor': '#2AD572'},
 {'name': 'Brentford',
  'shortName': 'Brentford',
  'id':

# TRANSFORM

In [11]:
rows = []
column_names = ['season','position','team_id','team','played', 'won', 'draw', 'lost', 'score_streak', 'goal_diff', 'points']


for club in standings:
    season          = 2025
    position        = club["idx"]
    team_id         = club["id"]
    team            = club["name"]
    played          = club["played"]
    won             = club["wins"]
    draw            = club["draws"]
    lost            = club["losses"]
    score_streak    = club["scoresStr"]
    goals_diff      = club["goalConDiff"]
    points          = club["pts"]

    tuple_of_club = (season, position, team_id, team, played, won, draw, lost, score_streak, goals_diff, points)
    
    rows.append(tuple_of_club)

In [44]:
# initializes the dataframe
teamData = pd.DataFrame(rows, columns=column_names)


In [ ]:
# teamData.head()

# LOAD

In [15]:
# Grabs credentials from env
PGHOST = os.getenv("host")
PGPORT = os.getenv("port")
PGUSER = os.getenv("username")
PGPASS = os.getenv("pass")
PGDB = os.getenv("dbname")

In [ ]:
# Creates a connection to the postgres database
conn = pg.connect(
    dbname= PGDB,
    user= PGUSER,
    password= PGPASS,
    host= PGHOST,
    port= PGPORT,
)
cursor = conn.cursor()
print(f"[SUCCESS] - connected to database")

[success] - connected to database


In [ ]:
# Roll backs existing transaction, checks for initial db connection
conn.rollback()
cursor.execute("SELECT 1")
print(f"[SUCCESS] - database connection verified: {cursor.fetchone()}")


[success] - database connection verified: (1,)


In [43]:
# Checks if the standings table exists inside db

sql_table = "standings"
cursor.execute(
    """
    SELECT EXISTS (
        SELECT FROM information_schema.tables
        WHERE table_schema = 'staging' AND table_name = %s
    )
    """,
    (sql_table,),
 )

if not cursor.fetchone()[0]:
    raise SystemExit(f"The table {sql_table} was not found")
else:
    print(f"[SUCCESS], the table {sql_table} exists")

OperationalError: the connection is closed

In [42]:
# Creates a dataframe reference (standings_df) for data upsert 

table_cols = ['season','position','team_id','team','played', 'won', 'draw', 'lost', 'score_streak', 'goal_diff', 'points']

standings_df = teamData[table_cols]

In [ ]:
# Used to assign each dataframe row into standard tuples (see last cell to unpack) 
standings_records_tuples = standings_df.itertuples(index= False, name= None)

# Used to store each tuple
list_standings_tuple = list(standings_records_tuples)

In [ ]:
# Data upsert query to insert records into postgre standings table 

UPSERT_SQL = """
INSERT INTO staging.standings (
season, position, team_id, team, played, won, draw, lost, score_streak, goal_diff, points

)
VALUES (%(season)s,%(position)s,%(team_id)s,%(team)s,%(played)s,%(won)s,%(draw)s,%(lost)s,%(score_streak)s,%(goal_diff)s,%(points)s)

ON CONFLICT (season, team_id) DO UPDATE SET

position        = EXCLUDED.position,
team            = EXCLUDED.team,
played          = EXCLUDED.played,
won             = EXCLUDED.won,
draw            = EXCLUDED.draw,
lost            = EXCLUDED.lost,
score_streak    = EXCLUDED.score_streak,
goal_diff       = EXCLUDED.goal_diff,
points          = EXCLUDED.points;
"""

In [26]:
num_of_updated_rows = len(list_standings_tuple)

In [ ]:
# used to send upsert query of all values inside list (the tuples) and commits rows
try:
    cursor.executemany(UPSERT_SQL, list_standings_tuple)
    conn.commit()
    print(f"[SUCCESS] UPSERT added {num_of_updated_rows}")

# error block and rollback for issues with try block
except Exception as e:
    conn.rollback()
    print(f"[ERROR] rollback caused by:  {e}")
    raise

# closes cursor and db connection
finally:
    cursor.close()
    conn.close()
    print("connection closed")

[SUCCESS] UPSERT added 20
connection closed


In [ ]:
# quick checks for data quality and formatting:

# regular checks across etl:
# print(rows)
# teamData.head()
# teamData.info()

# for response data:
# formatted_standings = json.dumps(standings, indent= 4) 
# print(f"\n\n\n{formatted_standings}")

# for checking duplicated season or position values:
# teamData[teamData.duplicated(["season", "position"], keep=False)]

# for checking/unpacking tuple data @ cell 40:  
# for ob in list_standings_tuple:
#   print(ob)